# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Week 8 Order of play

DAY 1: Modal and the SpecialistAgent  
DAY 2: RAG with Qdrant, FrontierAgent, Ensemble  
DAY 3: ScannerAgent, MessagingAgent  
DAY 4: The planning agent  
DAY 5: The user interface  

## DAY 4: the planning agent

Today one agent runs the whole loop: find deals, price each one, and tell you about the best. This follows `week8/day4.ipynb`.

We build it two ways, as Ed did:

1. **An autonomous agent.** GPT-5.1 is given three tools, and decides for itself when to call each.
2. **A fixed workflow.** Plain Python calls the same agents in a set order.

## When is a deal worth a message?

Ed sent an alert when the estimated value beat the price by more than \$50. That suited his products, which cost hundreds of dollars. Most of our wines cost \$10 to \$50, so a \$50 gap would almost never happen. We alert when the estimate is **at least 30% above the price, and at least \$10 more**. Both numbers are constants at the top of `agents/planning_agent.py`.

## Costs and time

| Step | Cost | Time |
|---|---|---|
| The practice run with pretend tools | under 1 cent | seconds |
| One full run of the autonomous agent | about 5 cents, plus a little Modal credit | about a minute |
| One full run of the fixed workflow | about 4 cents, plus a little Modal credit | about a minute and a half |

The first full run also waits about a minute while Modal wakes the fine-tuned model. Each full run can send one Telegram message to your phone.

In [ ]:
import json
import logging
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from agents.scanner_agent import ScannerAgent

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.1"

# Practice first, with pretend tools

Before handing the model real agents, we give it three pretend functions that do nothing but print. That way we can watch how it decides what to call, without paying for scans and estimates.

`test_scan()` returns three fixed deals without reading any website.

In [ ]:
test_results = ScannerAgent().test_scan()
test_results

In [ ]:
def scan_the_internet_for_bargains() -> str:
    """ This tool scans the internet for great wine deals and gets a curated list of promising deals """
    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    return test_results.model_dump_json()

In [ ]:
def estimate_true_value(description: str) -> str:
    """ This tool estimates the true value of a wine based on a text description of it """
    print(f"Fake function to estimate the true value of {description[:30]}... - this always returns $60")
    return f"Wine {description} has an estimated true value of $60"

In [ ]:
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """ This tool notifies the user of a great deal, given a description of it, the price of the deal, and the estimated true value """
    print(f"Fake function to notify the user of {description[:40]}... which costs {deal_price} and is estimated at {estimated_true_value}")
    return "notification sent ok"

In [ ]:
notify_user_of_deal("a Napa Cabernet", 25, 60, "https://www.example.com")

## Describe the tools to the model

The model can't see our Python. It only sees a description of each tool, written in JSON: its name, what it does, and what arguments it takes. The model reads these descriptions to decide which tool to call, so they matter as much as a prompt does.

In [ ]:
scan_function = {
    "name": "scan_the_internet_for_bargains",
    "description": "Returns top wine bargains scraped from the internet along with the price each wine is being offered for, in US dollars",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of a wine, estimate how much it is actually worth in US dollars",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the wine to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a message about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the wine itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            },
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            },
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": scan_function},
    {"type": "function", "function": estimate_function},
    {"type": "function", "function": notify_function},
]

## Run the tools the model asks for

When the model wants a tool, its reply says which one and with what arguments, instead of giving an answer. This function runs each requested tool and packages the result to send back.

In [ ]:
def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [ ]:
system_message = "You find great deals on bottles of wine using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain wine deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Only notify the user if the estimated true value is at least 30% above the price and at least $10 more. If no deal meets that, don't notify.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

## The agent loop

This short loop is the core of an agent. Ask the model; if it wants tools, run them, add the results to the conversation, and ask again. Stop when it answers instead of asking for a tool.

Watch the printed lines: the model scans once, estimates each of the three deals, then picks one to notify you about. The pretend estimate is always \$60, so it should choose the cheapest wine.

In [ ]:
done = False
while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

# Now with the real agents

`AutonomousPlanningAgent` in `agents/autonomous_planning_agent.py` is the same loop, with the pretend functions replaced by the real agents:

- **scan** calls the ScannerAgent from day 3
- **estimate** calls the EnsembleAgent from day 2, which asks the frontier model with RAG, your fine-tuned model on Modal, and the neural network
- **notify** asks the MessagingAgent to have Claude write a message and send it to Telegram

It needs the vector store, so we open it first. Only one program can have it open at a time: if you get an "already accessed" error, close the kernel of any other notebook using it.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
client = QdrantClient(path="wines_vectorstore")

In [ ]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent

agent = AutonomousPlanningAgent(client)

**The next cell is a full real run.** It reads today's deals, prices each one with three models, and, if one clears the bar, sends you a Telegram message about it.

When I tested it, it chose the Decades 5 Napa Cabernet from Last Bottle: offered at \$49, estimated by the ensemble at about \$140. The shop itself gives the retail price as \$175. Today's deals may be different.

In [ ]:
opportunity = agent.plan()
opportunity

# The fixed workflow

`PlanningAgent` in `agents/planning_agent.py` does the same job without letting a model decide anything: scan, price every deal, sort by discount, and alert about the best if it clears the bar.

Which is better? The fixed version is predictable and cheaper, and easier to test. The autonomous one is more flexible: give it a new tool and it can work out how to use it, without anyone changing its code. Ed's final app used the fixed version.

This run also prices five deals and can send one message.

In [ ]:
from agents.planning_agent import PlanningAgent

planner = PlanningAgent(client)
best = planner.plan()
best

## Before you leave

Close the vector store, so that tomorrow's app can open it.

In [ ]:
client.close()